Current format check

In [7]:
from core.file_manager import preprocess_file_manager

from settings.main_settings import test_settings

In [8]:
settings = test_settings().get_setting_dictionary()
preprocessed_steps = settings['preprocessed_steps']
preprocessing_steps_list = settings['preprocessing_steps_list']
channels = settings['channels']
original_data_folder = settings['original_data_folder']
target_spacing = settings['target_spacing']
crop_size = settings['crop_size']

patients = ['3322','001','003']


file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

In [9]:
last_step = preprocessed_steps[preprocessing_steps_list[-1][0]]['end']

In [10]:
patient = file_manager.get_file_names()[0]
patients = file_manager.get_file_names()
patient_data = file_manager.load_file_pickle(last_step,'3322')

In [11]:
patient_data.keys()

dict_keys(['adc', 'anatomy', 'dwi', 't2', 'lesion'])

In [12]:
dataset = {}


for patient in patients:
    patient_data = file_manager.load_file_pickle(last_step,patient)
    patiend_data_shard = {
        'adc': patient_data['adc'],
        'dwi': patient_data['dwi'],
        't2': patient_data['t2'],
        'anatomy': patient_data['anatomy'],

        'label': 1
        'spacing':

    }

SyntaxError: invalid syntax (2539811084.py, line 13)

Output that i need ... 

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import SimpleITK as sitk


def prepare_dataset(dataset, pp_dir):
    """
    Converts a dataset into the format expected by the loader.

    Parameters
    ----------
    dataset : dict
        {
            pid : {
                'adc': ndarray or sitk.Image,
                'dwi': ndarray or sitk.Image,
                't2': ndarray or sitk.Image,
                'anatomy': ndarray or sitk.Image,
                'label': int,
                'spacing': tuple (optional)
            }
        }

    pp_dir : str
        Output directory.
    """

    os.makedirs(pp_dir, exist_ok=True)

    info_rows = []

    for pid, sample in dataset.items():

        def to_numpy(x):
            """Convert SITK image to numpy if necessary."""
            if isinstance(x, sitk.Image):
                arr = sitk.GetArrayFromImage(x)      # (z,y,x)
                arr = np.transpose(arr, (2,1,0))     # -> (x,y,z)
                spacing = x.GetSpacing()
            else:
                arr = x
                spacing = None
            return arr, spacing

        adc, spacing_adc = to_numpy(sample["adc"])
        dwi, spacing_dwi = to_numpy(sample["dwi"])
        t2, spacing_t2 = to_numpy(sample["t2"])
        seg, spacing_seg = to_numpy(sample["anatomy"])

        # use spacing from image if available
        spacing = sample.get("spacing", spacing_adc)

        #########################################
        # image tensor (H,W,D,C)
        #########################################

        img = np.stack([
            adc.astype(np.float32),
            dwi.astype(np.float32),
            t2.astype(np.float32),
        ], axis=-1)

        #########################################
        # segmentation (H,W,D,1)
        #########################################

        seg = seg.astype(np.uint8)[..., None]

        #########################################
        # save arrays
        #########################################

        np.save(os.path.join(pp_dir, f"{pid}_img.npy"), img)
        np.save(os.path.join(pp_dir, f"{pid}_rois.npy"), seg)

        #########################################
        # foreground slices
        #########################################

        fg_slices = np.where(seg[...,0].sum(axis=(0,1)) > 0)[0].tolist()

        #########################################
        # meta info
        #########################################

        meta = {
            "pid": pid,
            "class_target": sample["label"],
            "spacing": spacing,
            "fg_slices": fg_slices,
        }

        with open(os.path.join(pp_dir, f"{pid}_meta_info.pickle"), "wb") as f:
            pickle.dump(meta, f)

        info_rows.append(meta)

    #############################################
    # info_df.pickle
    #############################################

    df = pd.DataFrame(info_rows)
    df.to_pickle(os.path.join(pp_dir, "info_df.pickle"))

    print(f"Saved {len(df)} patients.")